# 18 — Distributed training on two local CPU processes

**Level:** Advanced / expert practice · **Study time:** 90–120 minutes (including experiments)

**Prerequisites:** Notebooks 06, 15, and 17; keep the companion examples folder with this notebook.

## What you will be able to do
- Explain ranks, data sharding, and gradient synchronization.
- Launch an actual two-process DDP training job.
- Verify that both replicas finish with identical parameters.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 17](17_performance.ipynb) · [Next: 19](19_capstone.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Inspect how ranks receive different examples

In data parallel training, each process (rank) holds a copy of the model and processes a shard of data. DDP averages gradients so replicas make the same update. Equal local batch sizes make that average correspond to a global example mean. Our dataset divides exactly across ranks, avoiding sampler padding or dropped examples.


In [ ]:
from torch.utils.data.distributed import DistributedSampler
toy_dataset = TensorDataset(torch.arange(12))
shards = []
for rank in range(2):
    sampler = DistributedSampler(toy_dataset, num_replicas=2, rank=rank, shuffle=True, seed=SEED)
    sampler.set_epoch(0)
    shard = list(sampler)
    shards.append(shard)
    print('Rank', rank, 'indices:', shard)
assert set(shards[0]).isdisjoint(shards[1])
assert set(shards[0] + shards[1]) == set(range(12))


## Step 3 — Read the real worker program

Windows launches fresh Python interpreters for spawned workers. Put worker functions in a script and protect process creation with `if __name__ == '__main__'`; defining the worker only inside a notebook is unreliable. The companion script uses the Gloo backend, a file rendezvous, 2 CPU processes, and destroys the process group in a finally block. Inspect `sampler.set_epoch(epoch)`, `all_reduce`, and the rank-0-only output.


In [ ]:
from IPython.display import Code, display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'examples' / 'ddp_train.py').exists())
script = root / 'examples' / 'ddp_train.py'
display(Code(script.read_text(encoding='utf-8'), language='python'))


## Step 4 — Launch and verify the distributed run

This launches two short-lived local workers and normally takes seconds to a minute. It demonstrates synchronization, not a speed benefit. Small CPU jobs usually run slower with DDP overhead. If Gloo is absent, the branch reports a skip; use a PyTorch build with Gloo to complete the process experiment.


In [ ]:
import json
import subprocess
import sys
import torch.distributed as dist
RUN_DDP = True
if RUN_DDP and dist.is_available() and dist.is_gloo_available():
    output_dir = root / 'artifacts' / '18_ddp'
    result = subprocess.run([sys.executable, str(script), '--output-dir', str(output_dir)],
                            cwd=str(root), text=True, capture_output=True, timeout=180)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(result.stderr)
    report = json.loads((output_dir / 'report.json').read_text())
    print(report)
    assert report['world_size'] == 2
    assert report['replica_max_difference'] < 1e-6
    assert report['final_mse'] < 0.03
else:
    print('Two-process training skipped: RUN_DDP=False or Gloo is unavailable.')


## Your turn — practice before opening the solutions

1. Demonstrate that calling `set_epoch(1)` changes a shuffled sampler's order.
2. Calculate the global batch size when each of two ranks processes 16 examples.
3. Simulate averaging two local gradients and verify it matches the gradient of a combined equally sized batch.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
sampler = DistributedSampler(toy_dataset, num_replicas=2, rank=0, shuffle=True, seed=SEED)
sampler.set_epoch(0); first = list(sampler)
sampler.set_epoch(1); second = list(sampler)
assert first != second
print('Global batch size:', 2 * 16)
w = torch.tensor(0., requires_grad=True)
local_targets = [torch.tensor([1., 2.]), torch.tensor([3., 4.])]
local_grads = [torch.autograd.grad((w - targets).square().mean(), w)[0] for targets in local_targets]
combined_grad = torch.autograd.grad((w - torch.cat(local_targets)).square().mean(), w)[0]
assert torch.allclose(torch.stack(local_grads).mean(), combined_grad)


## Before moving on

DDP does not shard the data for you: use a sampler. All ranks must participate in collectives in a compatible order. On multiple CUDA GPUs, the usual architecture is one process per GPU with an appropriate supported backend; this local Gloo example does not test multi-GPU or multi-node behavior.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Getting started with DDP](https://docs.pytorch.org/tutorials/intermediate/ddp_tutorial.html)
- [DistributedSampler](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.distributed.DistributedSampler)


[Course guide](../README.md) · [Previous: 17](17_performance.ipynb) · [Next: 19](19_capstone.ipynb)
